# Пример использования calsyn

Этот ноутбук показывает базовый сценарий работы с `calsyn`: подготовка входного `df`, выделение `X` и `y`, обучение генератора, симуляция с заданным эффектом `tau` и диагностика результата.

`τ` — это эффект на целевую переменную, который мы задаем самостоятельно. Модель его не оценивает, а просто добавляет к синтетической траектории в выбранный период воздействия.

## 1. Импорт библиотек

In [ ]:
from datetime import date, timedelta

import matplotlib.pyplot as plt
import numpy as np
import polars as pl

from calsyn import CalibratedGenerator

## 2. Создаем примерный `df`

В реальном проекте вместо этого блока обычно будет загрузка ваших данных. Главное требование: `df` должен содержать колонку с датой, числовые признаки и одну целевую колонку.

In [ ]:
rng = np.random.default_rng(42)
n = 500
start = date(2023, 1, 1)
dates = [start + timedelta(days=i) for i in range(n)]

# Признаки: условные лог-доходности рыночных факторов.
gazp = rng.normal(0.0002, 0.018, n)
lkoh = rng.normal(0.0001, 0.016, n)
usdrub = rng.normal(0.0000, 0.010, n)

# Целевая переменная SBER зависит от признаков нелинейно плюс шум.
sber = (
    0.35 * gazp
    + 0.25 * lkoh
    - 0.15 * usdrub
    + 0.4 * gazp * lkoh
    + rng.normal(0.0, 0.012, n)
)

df = pl.DataFrame(
    {
        "date": dates,
        "GAZP": gazp,
        "LKOH": lkoh,
        "USDRUB": usdrub,
        "SBER": sber,
    }
)

df.head()

## 3. Готовим `X` и `y`

`X` — это дата и признаки. Целевая колонка в `X` не попадает. `y` — массив значений целевой переменной в том же порядке строк.

In [ ]:
target_col = "SBER"
date_col = "date"

X = df.drop(target_col)
y = df[target_col].to_numpy()

assert len(X) == len(y)
assert date_col in X.columns
assert target_col not in X.columns

X.head(), y[:5]

## 4. Обучаем генератор

`CalibratedGenerator` обучает CatBoost-модель `f(X)` и подбирает распределение шума по остаткам. Для примера зададим небольшое число итераций CatBoost, чтобы ноутбук выполнялся быстрее.

In [ ]:
gen = CalibratedGenerator(
    date_col=date_col,
    noise="auto",
    random_seed=42,
    catboost_params={
        "iterations": 300,
        "learning_rate": 0.05,
        "depth": 4,
        "loss_function": "RMSE",
        "verbose": 0,
    },
)

gen.fit(X, y)

print("OOS metrics:", gen.oos_metrics)
print("Noise params:", gen.noise_params)

## 5. Генерируем синтетические траектории с `tau`

Здесь `tau=0.02` — самостоятельно заданный эффект на целевую переменную. Он начинает добавляться с даты `treatment_start`.

In [ ]:
result = gen.generate(
    X,
    treatment_start="2024-01-01",
    tau=0.02,
    n_simulations=500,
)

print("simulations shape:", result.simulations.shape)
print("effect shape:", result.effect.shape)
print("f_X shape:", result.f_X.shape)

## 6. Визуализируем результат

In [ ]:
synth_mean = result.simulations.mean(axis=0)
lower = np.quantile(result.simulations, 0.05, axis=0)
upper = np.quantile(result.simulations, 0.95, axis=0)
date_values = df[date_col].to_list()

plt.figure(figsize=(12, 5))
plt.plot(date_values, y, label="real target", linewidth=1.5)
plt.plot(date_values, synth_mean, label="synthetic mean", linewidth=1.5)
plt.fill_between(date_values, lower, upper, alpha=0.2, label="90% interval")
plt.axvline(date(2024, 1, 1), color="black", linestyle="--", linewidth=1, label="treatment start")
plt.title("Synthetic trajectories with user-defined tau")
plt.legend()
plt.tight_layout()
plt.show()

## 7. Диагностика baseline без эффекта

Для диагностики удобно сгенерировать сценарий `tau=0.0`: это проверка того, насколько синтетические траектории похожи на реальные данные без добавочного эффекта.

In [ ]:
baseline = gen.generate(
    X,
    treatment_start="2024-01-01",
    tau=0.0,
    n_simulations=500,
)

report = gen.diagnose(baseline)

print("OOS metrics:", report.oos_metrics)
print("KS statistic:", report.ks.statistic)
print("KS p-value:", report.ks.pvalue)
print("Coverage 1 sigma:", report.coverage.coverage_1sigma)
print("Coverage 2 sigma:", report.coverage.coverage_2sigma)

## 8. Сетка эффектов `tau`

Можно передать список значений `tau`, чтобы сравнить несколько сценариев воздействия.

In [ ]:
tau_results = gen.generate(
    X,
    treatment_start="2024-01-01",
    tau=[0.0, 0.01, 0.02, 0.05],
    n_simulations=300,
)

for tau_value, tau_result in tau_results.items():
    print(f"tau={tau_value:0.3f}: synthetic mean={tau_result.simulations.mean():0.5f}")

## 9. Шум в признаках

Отдельный режим позволяет проверить, как пользовательски заданный шум в отдельных признаках влияет на целевую переменную.

In [ ]:
feature_noise_result = gen.generate_with_feature_noise(
    X,
    feature_noise={
        "USDRUB": {"distribution": "normal", "scale": 0.01},
        "GAZP": {"distribution": "t", "scale": 0.02, "df": 4},
    },
    n_simulations=300,
)

print("Y simulations:", feature_noise_result.simulations.shape)
print("Noisy USDRUB:", feature_noise_result.X_simulations["USDRUB"].shape)
print("f(X) mean shape:", feature_noise_result.f_X_mean.shape)